IMPORTING LIBRARIES

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

LOAD DATASET

In [3]:
df = pd.read_csv("../data/raw/bankloan.csv")

df.head()

,ID,Age,Experience,Income,ZIP.Code,Family,CCAvg,Education,Mortgage,Personal.Loan,Securities.Account,CD.Account,Online,CreditCard
0,1,25,1,49,91107,4,1.6,1,0,0,1,0,0,0
1,2,45,19,34,90089,3,1.5,1,0,0,1,0,0,0
2,3,39,15,11,94720,1,1.0,1,0,0,0,0,0,0
3,4,35,9,100,94112,1,2.7,2,0,0,0,0,0,0
4,5,35,8,45,91330,4,1.0,2,0,0,0,0,0,1


CHECKING SIZE OF THE DATASET

In [4]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 5000
Number of columns: 14


COLUMNS AND DATATYPES

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID                  5000 non-null   int64  
 1   Age                 5000 non-null   int64  
 2   Experience          5000 non-null   int64  
 3   Income              5000 non-null   int64  
 4   ZIP.Code            5000 non-null   int64  
 5   Family              5000 non-null   int64  
 6   CCAvg               5000 non-null   float64
 7   Education           5000 non-null   int64  
 8   Mortgage            5000 non-null   int64  
 9   Personal.Loan       5000 non-null   int64  
 10  Securities.Account  5000 non-null   int64  
 11  CD.Account          5000 non-null   int64  
 12  Online              5000 non-null   int64  
 13  CreditCard          5000 non-null   int64  
dtypes: float64(1), int64(13)
memory usage: 547.0 KB


CHECKING FOR MISSING VALUES

In [6]:
missing_values = df.isnull().sum()

print(missing_values)

ID                    0
Age                   0
Experience            0
Income                0
ZIP.Code              0
Family                0
CCAvg                 0
Education             0
Mortgage              0
Personal.Loan         0
Securities.Account    0
CD.Account            0
Online                0
CreditCard            0
dtype: int64


CHECKING FOR DUPLICATES

In [7]:
duplicates = df.duplicated().sum()

print("Number of duplicate rows:", duplicates)

Number of duplicate rows: 0


CHECK TARGET VARIABLE

In [8]:
print("Loan status distribution:")
print(df["Personal.Loan"].value_counts())

Loan status distribution:
Personal.Loan
0    4520
1     480
Name: count, dtype: int64


CHECK PERCENTAGE OF EACH CLASS

In [9]:
print("Loan status percentage:")
print(df["Personal.Loan"].value_counts(normalize=True) * 100)

Loan status percentage:
Personal.Loan
0    90.4
1     9.6
Name: proportion, dtype: float64


SEPARATE FEATURES AND TARGET

In [10]:
X = df.drop("Personal.Loan", axis=1)
y = df["Personal.Loan"]

print("Features:")
print(X.columns.tolist())

print("\nTarget:")
print(y.name)

Features:
['ID', 'Age', 'Experience', 'Income', 'ZIP.Code', 'Family', 'CCAvg', 'Education', 'Mortgage', 'Securities.Account', 'CD.Account', 'Online', 'CreditCard']

Target:
Personal.Loan


CHECK THE FEATURE DATA

In [11]:
X.head()

,ID,Age,Experience,Income,ZIP.Code,Family,CCAvg,Education,Mortgage,Securities.Account,CD.Account,Online,CreditCard
0,1,25,1,49,91107,4,1.6,1,0,1,0,0,0
1,2,45,19,34,90089,3,1.5,1,0,1,0,0,0
2,3,39,15,11,94720,1,1.0,1,0,0,0,0,0
3,4,35,9,100,94112,1,2.7,2,0,0,0,0,0
4,5,35,8,45,91330,4,1.0,2,0,0,0,0,1


In [12]:
X.dtypes

ID                      int64
Age                     int64
Experience              int64
Income                  int64
ZIP.Code                int64
Family                  int64
CCAvg                 float64
Education               int64
Mortgage                int64
Securities.Account      int64
CD.Account              int64
Online                  int64
CreditCard              int64
dtype: object

CHECK TARGET AND FEATURES TOGETHER

In [13]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (5000, 13)
y shape: (5000,)


SPLIT DATA

In [14]:
# Set a random seed so we get the same split every time
np.random.seed(42)

# Create shuffled row indices
indices = np.random.permutation(len(df))

# Calculate the size of the training set
train_size = int(0.8 * len(df))

# Split the indices
train_indices = indices[:train_size]
test_indices = indices[train_size:]

# Create training and testing sets
X_train = X.iloc[train_indices].reset_index(drop=True)
X_test = X.iloc[test_indices].reset_index(drop=True)

y_train = y.iloc[train_indices].reset_index(drop=True)
y_test = y.iloc[test_indices].reset_index(drop=True)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (4000, 13)
Testing data: (1000, 13)


CHECK CLASSES IN THE TRAINING DATA


In [15]:
print("Testing target distribution:")
print(y_test.value_counts())

Testing target distribution:
Personal.Loan
0    909
1     91
Name: count, dtype: int64


CALCULATE THE GINI IMPURITY

In [16]:
def gini_impurity(y):
    """
    Calculate the Gini impurity of a set of target values.
    """
    if len(y) == 0:
        return 0
    
    probabilities = y.value_counts(normalize=True)
    
    gini = 1 - np.sum(probabilities ** 2)
    
    return gini

testing it on our training target:

In [17]:
gini_root = gini_impurity(y_train)

print("Gini impurity of the root:", gini_root)

Gini impurity of the root: 0.17558487499999997


Verifying the Gini Calculation manually

In [18]:
class_counts = y_train.value_counts()

total = len(y_train)

p0 = class_counts.get(0, 0) / total
p1 = class_counts.get(1, 0) / total

gini = 1 - (p0**2 + p1**2)

print("Class 0 probability:", p0)
print("Class 1 probability:", p1)
print("Gini impurity:", gini)

Class 0 probability: 0.90275
Class 1 probability: 0.09725
Gini impurity: 0.17558487499999997


ACTUAL CLASS COUNTS


In [ ]:
class_counts = y_train.value_counts()

print(class_counts)

CALCULATE THE PROBABILITIES

In [19]:
total = len(y_train)

p0 = class_counts[0] / total
p1 = class_counts[1] / total

print("Probability of class 0:", p0)
print("Probability of class 1:", p1)

Probability of class 0: 0.90275
Probability of class 1: 0.09725


CALCULATE GINI MANUALLY

In [20]:
gini = 1 - (p0**2 + p1**2)

print("Root Gini Impurity:", gini)

Root Gini Impurity: 0.17558487499999997


CREATE A REUSABLE GINI FUNCTION

In [24]:
def gini_impurity(y):
    if len(y) == 0:
        return 0
    
    class_probabilities = y.value_counts(normalize=True)
    
    return 1 - np.sum(class_probabilities ** 2)

TEST FIRST DECISION TREE QUESTION

In [25]:
threshold = 50

left_group = y_train[X_train["Income"] <= threshold]
right_group = y_train[X_train["Income"] > threshold]

print("Left group size:", len(left_group))
print("Right group size:", len(right_group))

print("\nLeft group Gini:", gini_impurity(left_group))
print("Right group Gini:", gini_impurity(right_group))

Left group size: 1514
Right group size: 2486

Left group Gini: 0.0
Right group Gini: 0.2639828898630401


CALCULATE THE WEIGHTED GINI

In [ ]:
left_weight = len(left_group) / len(y_train)
right_weight = len(right_group) / len(y_train)

weighted_gini = (
    left_weight * gini_impurity(left_group)
    + right_weight * gini_impurity(right_group)
)

print("Left weight:", left_weight)
print("Right weight:", right_weight)
print("Weighted Gini:", weighted_gini)

COMPARE BEFORE AND AFTER SPLITING

In [ ]:
print("Gini before split:", gini_root)
print("Gini after split:", weighted_gini)

if weighted_gini < gini_root:
    print("The split improves purity.")
else:
    print("The split does not improve purity.")

CALCULATE GINI GAIN

In [ ]:
gini_gain = gini_root - weighted_gini

print("Gini Gain:", gini_gain)

TRYING ANOTHER THREESHOLD

In [27]:
threshold = 100

left_group = y_train[X_train["Income"] <= threshold]
right_group = y_train[X_train["Income"] > threshold]

weighted_gini = (
    (len(left_group) / len(y_train)) * gini_impurity(left_group)
    + (len(right_group) / len(y_train)) * gini_impurity(right_group)
)

gini_gain = gini_root - weighted_gini

print("Threshold:", threshold)
print("Weighted Gini:", weighted_gini)
print("Gini Gain:", gini_gain)

Threshold: 100
Weighted Gini: 0.1301020931200393
Gini Gain: 0.04548278187996066


TRYING MULTIPLE INCOME THRESHOLDS

In [28]:
thresholds = [20, 30, 40, 50, 60, 70, 80, 90, 100]

for threshold in thresholds:
    left_group = y_train[X_train["Income"] <= threshold]
    right_group = y_train[X_train["Income"] > threshold]

    weighted_gini = (
        (len(left_group) / len(y_train)) * gini_impurity(left_group)
        + (len(right_group) / len(y_train)) * gini_impurity(right_group)
    )

    gain = gini_root - weighted_gini

    print(
        "Threshold:", threshold,
        "| Weighted Gini:", round(weighted_gini, 4),
        "| Gini Gain:", round(gain, 4)
    )

Threshold: 20 | Weighted Gini: 0.174 | Gini Gain: 0.0016
Threshold: 30 | Weighted Gini: 0.1716 | Gini Gain: 0.004
Threshold: 40 | Weighted Gini: 0.1683 | Gini Gain: 0.0072
Threshold: 50 | Weighted Gini: 0.1641 | Gini Gain: 0.0115
Threshold: 60 | Weighted Gini: 0.1593 | Gini Gain: 0.0163
Threshold: 70 | Weighted Gini: 0.1537 | Gini Gain: 0.0219
Threshold: 80 | Weighted Gini: 0.146 | Gini Gain: 0.0296
Threshold: 90 | Weighted Gini: 0.1348 | Gini Gain: 0.0408
Threshold: 100 | Weighted Gini: 0.1301 | Gini Gain: 0.0455


PREPARE FEATURES

In [29]:
# Remove columns that should not be used as numerical decision features
X_model = X.drop(["ID", "ZIP.Code"], axis=1)

print("Features used for the model:")
print(X_model.columns.tolist())

Features used for the model:
['Age', 'Experience', 'Income', 'Family', 'CCAvg', 'Education', 'Mortgage', 'Securities.Account', 'CD.Account', 'Online', 'CreditCard']


UPDATE THE TRAINING AND TESTING DAA


In [30]:
X_train = X_model.iloc[train_indices].reset_index(drop=True)
X_test = X_model.iloc[test_indices].reset_index(drop=True)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (4000, 11)
Testing data: (1000, 11)


FUNCTION TO CREATE SPLIT'S GINI


In [31]:
def calculate_split_gini(X, y, feature, threshold):
    left = y[X[feature] <= threshold]
    right = y[X[feature] > threshold]

    if len(left) == 0 or len(right) == 0:
        return 1

    weighted_gini = (
        (len(left) / len(y)) * gini_impurity(left)
        + (len(right) / len(y)) * gini_impurity(right)
    )

    return weighted_gini

FIND BEST THRESHOLD FOR ONE FEATURE

In [34]:
feature = "Income"

values = sorted(X_train[feature].unique())

best_gini = 1
best_threshold = None

for threshold in values:
    weighted_gini = calculate_split_gini(
        X_train,
        y_train,
        feature,
        threshold
    )

    if weighted_gini < best_gini:
        best_gini = weighted_gini
        best_threshold = threshold

print("Feature:", feature)
print("Best threshold:", best_threshold)
print("Best weighted Gini:", best_gini)
print("Gini gain:", gini_root - best_gini)

Feature: Income
Best threshold: 114
Best weighted Gini: 0.12913514032288578
Gini gain: 0.04644973467711419


FUNCTION TO FIND BEST SPLIT

In [35]:
def find_best_split(X, y):
    best_feature = None
    best_threshold = None
    best_gini = 1

    for feature in X.columns:
        values = sorted(X[feature].unique())

        for threshold in values:
            weighted_gini = calculate_split_gini(
                X,
                y,
                feature,
                threshold
            )

            if weighted_gini < best_gini:
                best_gini = weighted_gini
                best_feature = feature
                best_threshold = threshold

    return best_feature, best_threshold, best_gini

FINDING THE BEST BEST SPLIT AT THE ROOT

In [36]:
best_feature, best_threshold, best_gini = find_best_split(
    X_train,
    y_train
)

print("Best feature:", best_feature)
print("Best threshold:", best_threshold)
print("Best weighted Gini:", best_gini)
print("Gini gain:", gini_root - best_gini)

Best feature: Income
Best threshold: 114
Best weighted Gini: 0.12913514032288578
Gini gain: 0.04644973467711419


CREATE FIRST TWO GROUPS

In [ ]:
left_mask = X_train[best_feature] <= best_threshold
right_mask = X_train[best_feature] > best_threshold

X_left = X_train[left_mask].reset_index(drop=True)
X_right = X_train[right_mask].resSet_index(drop=True)

y_left = y_train[left_mask].reset_index(drop=True)
y_right = y_train[right_mask].reset_index(drop=True)

print("Left group size:", len(y_left))
print("Right group size:", len(y_right))

print("\nLeft group Gini:", gini_impurity(y_left))
print("Right group Gini:", gini_impurity(y_right))

Left group size: 3203
Right group size: 797

Left group Gini: 0.041558759549590985
Right group Gini: 0.4810889014481847


CREATE A TREE NODE

In [38]:
class TreeNode:
    def __init__(self, feature=None, threshold=None, left=None, right=None, prediction=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.prediction = prediction

CREATE THE RECURSIVE TREE BULDING FUNCTION

In [48]:
def build_tree(X, y, depth=0, max_depth=3):
    
    # If all observations belong to one class
    if len(y.unique()) == 1:
        return TreeNode(prediction=y.iloc[0])
    
    # Stop if maximum depth is reached
    if depth >= max_depth:
        prediction = y.value_counts().idxmax()
        return TreeNode(prediction=prediction)
    
    # Find the best split
    feature, threshold, best_gini = find_best_split(X, y)
    
    # If no useful split exists
    if feature is None:
        prediction = y.value_counts().idxmax()
        return TreeNode(prediction=prediction)
    
    # Divide the data
    left_mask = X[feature] <= threshold
    right_mask = X[feature] > threshold
    
    X_left = X[left_mask]
    X_right = X[right_mask]
    
    y_left = y[left_mask]
    y_right = y[right_mask]
    
    # Build the branches recursively
    left_child = build_tree(
        X_left,
        y_left,
        depth + 1,
        max_depth
    )
    
    right_child = build_tree(
        X_right,
        y_right,
        depth + 1,
        max_depth
    )
    
    return TreeNode(
        feature=feature,
        threshold=threshold,
        left=left_child,
        right=right_child
    )

BUILD TREE

In [49]:
tree = build_tree(
    X_train,
    y_train,
    max_depth=3
)

print("Decision Tree built successfully.")

Decision Tree built successfully.


DISPLAY THE TREE STRUCTURE

In [46]:
def print_tree(node, depth=0):
    
    indentation = "    " * depth
    
    if node.prediction is not None:
        print(indentation + "Predict:", node.prediction)
        return
    
    print(
        indentation
        + f"If {node.feature} <= {node.threshold}:"
    )
    
    print_tree(node.left, depth + 1)
    
    print(
        indentation
        + "Else:"
    )
    
    print_tree(node.right, depth + 1)

PRINT DECISION TREE

In [50]:
print_tree(tree)

If Income <= 114:
    If CCAvg <= 2.9:
        If Income <= 105:
            Predict: 0
        Else:
            Predict: 0
    Else:
        If CD.Account <= 0:
            Predict: 0
        Else:
            Predict: 1
Else:
    If Education <= 1:
        If Family <= 2:
            Predict: 0
        Else:
            Predict: 1
    Else:
        If Income <= 115:
            Predict: 0
        Else:
            Predict: 1


MAKE PREDICTIONS

In [51]:
def predict_one(node, row):
    
    # If we reached a prediction node
    if node.prediction is not None:
        return node.prediction
    
    # Follow the correct branch
    if row[node.feature] <= node.threshold:
        return predict_one(node.left, row)
    else:
        return predict_one(node.right, row)

PREDICT THE ENTIRE TEST DATASET

In [52]:
def predict(tree, X):
    predictions = []
    
    for _, row in X.iterrows():
        prediction = predict_one(tree, row)
        predictions.append(prediction)
    
    return np.array(predictions)

In [53]:
y_pred = predict(tree, X_test)

print("First 20 predictions:")
print(y_pred[:20])

First 20 predictions:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


COMPARE PREDICTIONS WITH ACTUAL RESULTS

In [54]:
print("Actual values:")
print(y_test.values[:20])

print("\nPredicted values:")
print(y_pred[:20])

Actual values:
[0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]

Predicted values:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


CALCULATE ACCURACY MANUALLY

In [55]:
correct = np.sum(y_pred == y_test.values)
total = len(y_test)

accuracy = correct / total

print("Accuracy:", accuracy)
print("Accuracy (%):", accuracy * 100)

Accuracy: 0.978
Accuracy (%): 97.8


CONFUSION MATRIX

In [56]:
# Initialize confusion matrix values

TP = 0
TN = 0
FP = 0
FN = 0

for actual, predicted in zip(y_test.values, y_pred):
    
    if actual == 1 and predicted == 1:
        TP += 1
    
    elif actual == 0 and predicted == 0:
        TN += 1
    
    elif actual == 0 and predicted == 1:
        FP += 1
    
    elif actual == 1 and predicted == 0:
        FN += 1

print("True Positives:", TP)
print("True Negatives:", TN)
print("False Positives:", FP)
print("False Negatives:", FN)

True Positives: 71
True Negatives: 907
False Positives: 2
False Negatives: 20


DISPLAYING CONFUSION MATRIX

In [58]:
print("Confusion Matrix")
print()
print("                Predicted")
print("                0       1")
print(f"Actual  0      {TN:4}    {FP:4}")
print(f"        1      {FN:4}    {TP:4}")

Confusion Matrix

                Predicted
                0       1
Actual  0       907       2
        1        20      71


CALCULATE PRECISION

In [59]:
precision = TP / (TP + FP)

print("Precision:", precision)
print("Precision (%):", precision * 100)

Precision: 0.9726027397260274
Precision (%): 97.26027397260275


CALCULATE RECALL

In [60]:
recall = TP / (TP + FN)

print("Recall:", recall)
print("Recall (%):", recall * 100)

Recall: 0.7802197802197802
Recall (%): 78.02197802197803


CALCULATE F1-SCORE

In [61]:
f1_score = 2 * (precision * recall) / (precision + recall)

print("F1-Score:", f1_score)
print("F1-Score (%):", f1_score * 100)

F1-Score: 0.8658536585365854
F1-Score (%): 86.58536585365853


ALL METRICS TOGETHER

In [62]:
print("Decision Tree Evaluation")
print("------------------------")
print(f"Accuracy :  {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Precision:  {precision:.4f} ({precision * 100:.2f}%)")
print(f"Recall   :  {recall:.4f} ({recall * 100:.2f}%)")
print(f"F1-Score :  {f1_score:.4f} ({f1_score * 100:.2f}%)")

Decision Tree Evaluation
------------------------
Accuracy :  0.9780 (97.80%)
Precision:  0.9726 (97.26%)
Recall   :  0.7802 (78.02%)
F1-Score :  0.8659 (86.59%)


SUMMARY TABLE

In [63]:
metrics = {
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Score": [accuracy, precision, recall, f1_score]
}

metrics_df = pd.DataFrame(metrics)

print(metrics_df)

      Metric     Score
0   Accuracy  0.978000
1  Precision  0.972603
2     Recall  0.780220
3   F1-Score  0.865854


RESULTS INTERPRETATION

In [64]:
print("Model Interpretation")
print("--------------------")

print(f"The Decision Tree achieved an accuracy of {accuracy * 100:.2f}%.")

print(
    f"Precision was {precision * 100:.2f}%, "
    f"meaning that this percentage of predicted positive cases were actually positive."
)

print(
    f"Recall was {recall * 100:.2f}%, "
    f"meaning that the model identified this percentage of the actual positive cases."
)

print(
    f"The F1-score was {f1_score * 100:.2f}%, "
    f"providing a balance between precision and recall."
)

Model Interpretation
--------------------
The Decision Tree achieved an accuracy of 97.80%.
Precision was 97.26%, meaning that this percentage of predicted positive cases were actually positive.
Recall was 78.02%, meaning that the model identified this percentage of the actual positive cases.
The F1-score was 86.59%, providing a balance between precision and recall.


FINAL SUMMARY

In [65]:
print("===================================")
print("   DECISION TREE FROM SCRATCH")
print("===================================")

print(f"Maximum Depth : 3")
print(f"Accuracy      : {accuracy * 100:.2f}%")
print(f"Precision     : {precision * 100:.2f}%")
print(f"Recall        : {recall * 100:.2f}%")
print(f"F1-Score      : {f1_score * 100:.2f}%")

print("\nConfusion Matrix:")
print(f"True Negatives : {TN}")
print(f"False Positives: {FP}")
print(f"False Negatives: {FN}")
print(f"True Positives : {TP}")

   DECISION TREE FROM SCRATCH
Maximum Depth : 3
Accuracy      : 97.80%
Precision     : 97.26%
Recall        : 78.02%
F1-Score      : 86.59%

Confusion Matrix:
True Negatives : 907
False Positives: 2
False Negatives: 20
True Positives : 71
